# 🎯 Matching Engine & Explainable AI (Lost & Found Matcher)
This notebook implements the core matching pipeline:
- Calculates text cosine similarity between descriptions
- Evaluates attribute matching (Category, Color, Location, Date Proximity)
- Runs inference with the trained **Logistic Regression** model
- Produces an explainable composite score and human-readable reasons


In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

log_reg = joblib.load("models/match_classifier.pkl")
print("Logistic Regression model loaded successfully.")


Logistic Regression model loaded successfully.


### Helper Functions: Text Similarity & Date Calculation


In [2]:
def calculate_date_difference(d1_str, d2_str):
    try:
        d1 = datetime.strptime(str(d1_str)[:10], "%Y-%m-%d")
        d2 = datetime.strptime(str(d2_str)[:10], "%Y-%m-%d")
        return abs((d1 - d2).days)
    except Exception:
        return 7

def compute_text_similarity(t1, t2):
    v = TfidfVectorizer(stop_words="english")
    try:
        m = v.fit_transform([t1, t2])
        return float(np.clip(cosine_similarity(m[0:1], m[1:2])[0][0], 0.0, 1.0))
    except Exception:
        return 0.0

print("Date diff test (2026-11-20 vs 2026-11-22):", calculate_date_difference("2026-11-20", "2026-11-22"), "days")
print("Text sim test:", round(compute_text_similarity("black leather wallet", "dark leather bi-fold wallet") * 100, 1), "%")


Date diff test (2026-11-20 vs 2026-11-22): 2 days
Text sim test: 35.6 %


### Matching Logic & Explainability Function


In [3]:
def match_reports(lost_report, found_report):
    text_sim = compute_text_similarity(lost_report["description"], found_report["description"])
    same_cat = 1 if lost_report["category"].lower() == found_report["category"].lower() else 0
    same_col = 1 if lost_report["color"].lower() == found_report["color"].lower() else 0
    same_loc = 1 if lost_report["location"].lower() == found_report["location"].lower() else 0
    date_diff = calculate_date_difference(lost_report["report_date"], found_report["report_date"])

    features_df = pd.DataFrame(
        [[text_sim, same_cat, same_col, same_loc, date_diff]],
        columns=["text_similarity", "same_category", "same_color", "same_location", "date_diff_days"]
    )

    pred = int(log_reg.predict(features_df)[0])
    prob = float(log_reg.predict_proba(features_df)[0][1])

    date_score = max(0.0, 1.0 - (date_diff / 14.0))
    composite_score = (
        (text_sim * 0.35) +
        (same_cat * 0.30) +
        (same_loc * 0.15) +
        (same_col * 0.10) +
        (date_score * 0.10)
    ) * 100.0

    reasons = []
    if same_cat: reasons.append(f"Matching category: Both are '{lost_report['category']}'")
    if same_loc: reasons.append(f"Same location: Reported at '{lost_report['location']}'")
    if same_col: reasons.append(f"Matching color: Both listed as '{lost_report['color']}'")
    if text_sim > 0.2: reasons.append(f"Text similarity: {int(text_sim*100)}% TF-IDF cosine match")
    if date_diff <= 3: reasons.append(f"Reported within {date_diff} day(s) of each other")

    return {
        "is_potential_match": bool(pred),
        "ml_probability": round(prob * 100, 1),
        "composite_score": round(composite_score, 1),
        "text_similarity": round(text_sim * 100, 1),
        "date_diff_days": date_diff,
        "reasons": reasons
    }

# Test match
sample_lost = {
    "item_name": "Black Leather Wallet", "category": "Wallet", "color": "Black",
    "location": "Central Library", "report_date": "2026-11-20",
    "description": "Black leather bi-fold wallet containing student ID card and cash."
}
sample_found = {
    "item_name": "Dark Leather Wallet", "category": "Wallet", "color": "Black",
    "location": "Central Library", "report_date": "2026-11-20",
    "description": "Found a black leather bi-fold wallet on 2nd floor library reading desk with student cards."
}

res = match_reports(sample_lost, sample_found)
print("Match Evaluation Result:")
for k, v in res.items():
    print(f"  {k}: {v}")


Match Evaluation Result:
  is_potential_match: True
  ml_probability: 98.4
  composite_score: 78.3
  text_similarity: 38.1
  date_diff_days: 0
  reasons: ["Matching category: Both are 'Wallet'", "Same location: Reported at 'Central Library'", "Matching color: Both listed as 'Black'", 'Text similarity: 38% TF-IDF cosine match', 'Reported within 0 day(s) of each other']
